<a href="https://colab.research.google.com/github/gurudattamanpreet/Practice/blob/main/Titanic4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_csv('https://raw.githubusercontent.com/gurudattamanpreet/datasets/refs/heads/main/Titanic.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [5]:
df.drop(['PassengerId','Name','Ticket','Cabin'],axis=1,inplace=True)

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [7]:
df.dtypes

,0
Survived,int64
Pclass,int64
Sex,object
Age,float64
SibSp,int64
Parch,int64
Fare,float64
Embarked,object


In [8]:
df.isna().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [9]:
# new way of filling missing values
df.fillna({'Age':df['Age'].mean(),'Embarked':df['Embarked'].mode()[0]},inplace=True)

In [10]:
df.isna().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [11]:
pd.DataFrame({'data_types':df.dtypes,'unique_values':df.nunique()})

,data_types,unique_values
Survived,int64,2
Pclass,int64,3
Sex,object,2
Age,float64,89
SibSp,int64,7
Parch,int64,7
Fare,float64,248
Embarked,object,3


In [12]:
df.select_dtypes(include=['int64','float64']).columns

Index(['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [13]:
s=df[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']]

Q1=s.quantile(0.25)
Q3=s.quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-(1.5*IQR)
upper_bound=Q3+(1.5*IQR)
lower_outlier=s<lower_bound
upper_outlier=s>upper_bound

In [14]:
print(lower_outlier.sum())

Pclass     0
Age       24
SibSp      0
Parch      0
Fare       0
dtype: int64


In [15]:
print(upper_outlier.sum())

Pclass      0
Age        42
SibSp      46
Parch     213
Fare      116
dtype: int64


In [16]:
s2=['Age','SibSp','Parch','Fare']
for i in s2:
  lb=lower_bound[i]
  ub=upper_bound[i]
  df[i]=df[i].clip(lower=lb,upper=ub)

In [17]:
df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,891.000000,891.0,891.000000
mean,0.383838,2.308642,29.376817,0.426487,0.0,24.046813
std,0.486592,0.836071,12.062035,0.708246,0.0,20.481625
min,0.000000,1.000000,2.500000,0.000000,0.0,0.000000
25%,0.000000,2.000000,22.000000,0.000000,0.0,7.910400
50%,0.000000,3.000000,29.699118,0.000000,0.0,14.454200
75%,1.000000,3.000000,35.000000,1.000000,0.0,31.000000
max,1.000000,3.000000,54.500000,2.500000,0.0,65.634400


In [18]:
df['Familysize']=df['SibSp']+df['Parch']+1
df['Alone']=(df['Familysize']==1).astype(int)
df['Fareperperson']=df['Fare']/df['Familysize']

In [19]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Familysize,Alone,Fareperperson
0,0,3,male,22.0,1.0,0,7.2500,S,2.0,0,3.6250
1,1,1,female,38.0,1.0,0,65.6344,C,2.0,0,32.8172
2,1,3,female,26.0,0.0,0,7.9250,S,1.0,1,7.9250
3,1,1,female,35.0,1.0,0,53.1000,S,2.0,0,26.5500
4,0,3,male,35.0,0.0,0,8.0500,S,1.0,1,8.0500


In [20]:
df['Sex']=df['Sex'].map({'male':1,'female':0})

In [21]:
df=pd.get_dummies(df,columns=['Embarked'],drop_first=True)

In [22]:
df.head(1)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Familysize,Alone,Fareperperson,Embarked_Q,Embarked_S
0,0,3,1,22.0,1.0,0,7.25,2.0,0,3.625,False,True


In [23]:
df.drop(['SibSp','Parch'],axis=1,inplace=True)

In [24]:
df.head()

,Survived,Pclass,Sex,Age,Fare,Familysize,Alone,Fareperperson,Embarked_Q,Embarked_S
0,0,3,1,22.0,7.2500,2.0,0,3.6250,False,True
1,1,1,0,38.0,65.6344,2.0,0,32.8172,False,False
2,1,3,0,26.0,7.9250,1.0,1,7.9250,False,True
3,1,1,0,35.0,53.1000,2.0,0,26.5500,False,True
4,0,3,1,35.0,8.0500,1.0,1,8.0500,False,True


In [25]:
df2=df.copy()

In [26]:
X=df.drop(['Survived'],axis=1)
y=df['Survived']

In [27]:
sc=StandardScaler()
X_Scaled=sc.fit_transform(X)

In [28]:
X_train,X_test,y_train,y_test=train_test_split(X_Scaled,y,test_size=0.2,random_state=1,stratify=y)

In [29]:
X_train.shape

(712, 9)

In [30]:
X_test.shape

(179, 9)

In [31]:
y_train.value_counts()

,count
Survived,
0,439
1,273


In [32]:
y_test.value_counts()

,count
Survived,
0,110
1,69


In [33]:
lg=LogisticRegression()
lg.fit(X_train,y_train)

LogisticRegression()

In [34]:
y_pred_train=lg.predict(X_train)
y_pred_test=lg.predict(X_test)

In [35]:
accuracy_score(y_test,y_pred_test)

0.8156424581005587

In [36]:
accuracy_score(y_train,y_pred_train)

0.8132022471910112

In [37]:
print(classification_report(y_test,y_pred_test))

              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179



In [38]:
df2.head()

,Survived,Pclass,Sex,Age,Fare,Familysize,Alone,Fareperperson,Embarked_Q,Embarked_S
0,0,3,1,22.0,7.2500,2.0,0,3.6250,False,True
1,1,1,0,38.0,65.6344,2.0,0,32.8172,False,False
2,1,3,0,26.0,7.9250,1.0,1,7.9250,False,True
3,1,1,0,35.0,53.1000,2.0,0,26.5500,False,True
4,0,3,1,35.0,8.0500,1.0,1,8.0500,False,True


In [39]:
X2=df2.drop(['Survived'],axis=1)
y2=df2['Survived']

In [41]:
X_train2,X_test2,y_train2,y_test2=train_test_split(X,y,test_size=0.2,random_state=1,stratify=y)

In [42]:
sc2=StandardScaler()
X_train_Scaled=sc.fit_transform(X_train)
X_test_Scaled=sc.transform(X_test)

In [44]:
lg2=LogisticRegression()
lg2.fit(X_train_Scaled,y_train)

LogisticRegression()

In [45]:
y_pred_train2=lg2.predict(X_train_Scaled)
y_pred_test2=lg2.predict(X_test_Scaled)

In [46]:
accuracy_score(y_test2,y_pred_test2)

0.8156424581005587

In [47]:
print(classification_report(y_test2,y_pred_test2))

              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179



In [48]:
print(classification_report(y_train2,y_pred_train2))

              precision    recall  f1-score   support

           0       0.84      0.87      0.85       439
           1       0.77      0.73      0.75       273

    accuracy                           0.81       712
   macro avg       0.80      0.80      0.80       712
weighted avg       0.81      0.81      0.81       712

